téléchargement de la dataset fusionnée 

In [1]:
import pandas as pd

df = pd.read_csv("cpu_full_dataset.csv")
df.head()


,timestamp,value,serveur_id
0,2014-02-14 14:30:00,0.132,24ae8d
1,2014-02-14 14:35:00,0.134,24ae8d
2,2014-02-14 14:40:00,0.134,24ae8d
3,2014-02-14 14:45:00,0.134,24ae8d
4,2014-02-14 14:50:00,0.134,24ae8d


Exercice A : indexation 
Transformation de la colonne timestamp en format datetime et de la définir comme index du DataFrame afin de pouvoir utiliser les fonctionnalités spécifiques aux séries temporelles

In [2]:

df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values(by=['serveur_id', 'timestamp'])
df = df.set_index('timestamp')
df.head()


,value,serveur_id
timestamp,,
2014-02-14 14:30:00,0.132,24ae8d
2014-02-14 14:35:00,0.134,24ae8d
2014-02-14 14:40:00,0.134,24ae8d
2014-02-14 14:45:00,0.134,24ae8d
2014-02-14 14:50:00,0.134,24ae8d


Resampling
Regroupement des données par heure et calcul de la moyenne d'utilisation du cpu par heure afin d'obtenir une vision plus globale 

In [3]:
df_hourly = (df.groupby('serveur_id')['value'].resample('h').mean())
df_hourly.head()


serveur_id  timestamp          
24ae8d      2014-02-14 14:00:00    0.133667
            2014-02-14 15:00:00    0.122333
            2014-02-14 16:00:00    0.122667
            2014-02-14 17:00:00    0.133667
            2014-02-14 18:00:00    0.128333
Name: value, dtype: float64

Rolling Window
Calcul de la moyenne mobile des 5 derniéres observations afin de mieux observer la tendances d'utilisation du cpu

In [4]:
df['moyenne_mobile_5'] = df.groupby('serveur_id')['value'].transform(lambda x: x.rolling(5).mean())
df.head(15)

,value,serveur_id,moyenne_mobile_5
timestamp,,,
2014-02-14 14:30:00,0.132,24ae8d,NaN
2014-02-14 14:35:00,0.134,24ae8d,NaN
2014-02-14 14:40:00,0.134,24ae8d,NaN
2014-02-14 14:45:00,0.134,24ae8d,NaN
2014-02-14 14:50:00,0.134,24ae8d,0.1336
2014-02-14 14:55:00,0.134,24ae8d,0.1340
2014-02-14 15:00:00,0.134,24ae8d,0.1340
2014-02-14 15:05:00,0.134,24ae8d,0.1340
2014-02-14 15:10:00,0.066,24ae8d,0.1204


Lagging
Création  d'une variable cible correspondant à la valeur future du CPU, afin de transformer la série temporelle en problème de régression supervisée pour la modélisation prédictive.

In [5]:
df['target'] = df.groupby('serveur_id')['value'].shift(-1)
df.head(15)


,value,serveur_id,moyenne_mobile_5,target
timestamp,,,,
2014-02-14 14:30:00,0.132,24ae8d,NaN,0.134
2014-02-14 14:35:00,0.134,24ae8d,NaN,0.134
2014-02-14 14:40:00,0.134,24ae8d,NaN,0.134
2014-02-14 14:45:00,0.134,24ae8d,NaN,0.134
2014-02-14 14:50:00,0.134,24ae8d,0.1336,0.134
2014-02-14 14:55:00,0.134,24ae8d,0.1340,0.134
2014-02-14 15:00:00,0.134,24ae8d,0.1340,0.134
2014-02-14 15:05:00,0.134,24ae8d,0.1340,0.066
2014-02-14 15:10:00,0.066,24ae8d,0.1204,0.132


les features qu'on peut ajouter pour améliorer la dataset pour machine learning 

Lag features à t-2:

In [6]:
df['lag(t-2)'] = df.groupby('serveur_id')['value'].shift(2)
df.head(15)


,value,serveur_id,moyenne_mobile_5,target,lag(t-2)
timestamp,,,,,
2014-02-14 14:30:00,0.132,24ae8d,NaN,0.134,NaN
2014-02-14 14:35:00,0.134,24ae8d,NaN,0.134,NaN
2014-02-14 14:40:00,0.134,24ae8d,NaN,0.134,0.132
2014-02-14 14:45:00,0.134,24ae8d,NaN,0.134,0.134
2014-02-14 14:50:00,0.134,24ae8d,0.1336,0.134,0.134
2014-02-14 14:55:00,0.134,24ae8d,0.1340,0.134,0.134
2014-02-14 15:00:00,0.134,24ae8d,0.1340,0.134,0.134
2014-02-14 15:05:00,0.134,24ae8d,0.1340,0.066,0.134
2014-02-14 15:10:00,0.066,24ae8d,0.1204,0.132,0.134


Rolling std: pour capturer la variabilité du cpu si l'ecart type est faible cpu stable et si l'ecart type est élevé cpu unstable 

In [9]:
df["rolling_std_5"] = df["value"].rolling(5).std()
df.head(15)

,value,serveur_id,moyenne_mobile_5,target,lag(t-2),rolling_std_5
timestamp,,,,,,
2014-02-14 14:30:00,0.132,24ae8d,NaN,0.134,NaN,NaN
2014-02-14 14:35:00,0.134,24ae8d,NaN,0.134,NaN,NaN
2014-02-14 14:40:00,0.134,24ae8d,NaN,0.134,0.132,NaN
2014-02-14 14:45:00,0.134,24ae8d,NaN,0.134,0.134,NaN
2014-02-14 14:50:00,0.134,24ae8d,0.1336,0.134,0.134,0.000894
2014-02-14 14:55:00,0.134,24ae8d,0.1340,0.134,0.134,0.000000
2014-02-14 15:00:00,0.134,24ae8d,0.1340,0.134,0.134,0.000000
2014-02-14 15:05:00,0.134,24ae8d,0.1340,0.066,0.134,0.000000
2014-02-14 15:10:00,0.066,24ae8d,0.1204,0.132,0.134,0.030411


Différence : permet de capturer les changements rapides et les pics soudains d'utilisation 

In [11]:
df["diff"] = df["value"].diff()
df.head(15)

,value,serveur_id,moyenne_mobile_5,target,lag(t-2),rolling_std_5,diff
timestamp,,,,,,,
2014-02-14 14:30:00,0.132,24ae8d,NaN,0.134,NaN,NaN,NaN
2014-02-14 14:35:00,0.134,24ae8d,NaN,0.134,NaN,NaN,0.002
2014-02-14 14:40:00,0.134,24ae8d,NaN,0.134,0.132,NaN,0.000
2014-02-14 14:45:00,0.134,24ae8d,NaN,0.134,0.134,NaN,0.000
2014-02-14 14:50:00,0.134,24ae8d,0.1336,0.134,0.134,0.000894,0.000
2014-02-14 14:55:00,0.134,24ae8d,0.1340,0.134,0.134,0.000000,0.000
2014-02-14 15:00:00,0.134,24ae8d,0.1340,0.134,0.134,0.000000,0.000
2014-02-14 15:05:00,0.134,24ae8d,0.1340,0.066,0.134,0.000000,0.000
2014-02-14 15:10:00,0.066,24ae8d,0.1204,0.132,0.134,0.030411,-0.068


Feature temporelle 
hour feature: pour capturer les patterns journaliers 
weekday feature:permet de capturer les patterns hebdomadaires

In [15]:
df = df.reset_index()
df["hour"] = df["timestamp"].dt.hour
df["weekday"] = df["timestamp"].dt.weekday
df.head(15)

,index,timestamp,value,serveur_id,moyenne_mobile_5,target,lag(t-2),rolling_std_5,diff,hour,weekday
0,0,2014-02-14 14:30:00,0.132,24ae8d,NaN,0.134,NaN,NaN,NaN,14,4
1,1,2014-02-14 14:35:00,0.134,24ae8d,NaN,0.134,NaN,NaN,0.002,14,4
2,2,2014-02-14 14:40:00,0.134,24ae8d,NaN,0.134,0.132,NaN,0.000,14,4
3,3,2014-02-14 14:45:00,0.134,24ae8d,NaN,0.134,0.134,NaN,0.000,14,4
4,4,2014-02-14 14:50:00,0.134,24ae8d,0.1336,0.134,0.134,0.000894,0.000,14,4
5,5,2014-02-14 14:55:00,0.134,24ae8d,0.1340,0.134,0.134,0.000000,0.000,14,4
6,6,2014-02-14 15:00:00,0.134,24ae8d,0.1340,0.134,0.134,0.000000,0.000,15,4
7,7,2014-02-14 15:05:00,0.134,24ae8d,0.1340,0.066,0.134,0.000000,0.000,15,4
8,8,2014-02-14 15:10:00,0.066,24ae8d,0.1204,0.132,0.134,0.030411,-0.068,15,4
9,9,2014-02-14 15:15:00,0.132,24ae8d,0.1200,0.134,0.134,0.030199,0.066,15,4


Nettoyage final

In [16]:
df=df.dropna()
df.head(15)

,index,timestamp,value,serveur_id,moyenne_mobile_5,target,lag(t-2),rolling_std_5,diff,hour,weekday
4,4,2014-02-14 14:50:00,0.134,24ae8d,0.1336,0.134,0.134,0.000894,0.000,14,4
5,5,2014-02-14 14:55:00,0.134,24ae8d,0.1340,0.134,0.134,0.000000,0.000,14,4
6,6,2014-02-14 15:00:00,0.134,24ae8d,0.1340,0.134,0.134,0.000000,0.000,15,4
7,7,2014-02-14 15:05:00,0.134,24ae8d,0.1340,0.066,0.134,0.000000,0.000,15,4
8,8,2014-02-14 15:10:00,0.066,24ae8d,0.1204,0.132,0.134,0.030411,-0.068,15,4
9,9,2014-02-14 15:15:00,0.132,24ae8d,0.1200,0.134,0.134,0.030199,0.066,15,4
10,10,2014-02-14 15:20:00,0.134,24ae8d,0.1200,0.066,0.066,0.030199,0.002,15,4
11,11,2014-02-14 15:25:00,0.066,24ae8d,0.1064,0.132,0.132,0.036889,-0.068,15,4
12,12,2014-02-14 15:30:00,0.132,24ae8d,0.1060,0.202,0.134,0.036524,0.066,15,4
13,13,2014-02-14 15:35:00,0.202,24ae8d,0.1332,0.068,0.066,0.048096,0.070,15,4
